In [ ]:
import torch
import numpy as np
import nibabel as nib
from scipy.ndimage import zoom

def load_and_predict(cbct_path, model_path='/content/drive/MyDrive/MMDental/models/best_model.pth'):
    """
    Load trained model and predict on a new CBCT scan
    """
    # Load model
    checkpoint = torch.load(model_path, map_location='cpu', weights_only=False)

    # Recreate model architecture
    class Simple3DCNN(torch.nn.Module):
        def __init__(self, num_classes=8):
            super().__init__()
            self.conv_layers = torch.nn.Sequential(
                torch.nn.Conv3d(1, 32, kernel_size=3, padding=1),
                torch.nn.BatchNorm3d(32),
                torch.nn.ReLU(inplace=True),
                torch.nn.MaxPool3d(2),
                torch.nn.Conv3d(32, 64, kernel_size=3, padding=1),
                torch.nn.BatchNorm3d(64),
                torch.nn.ReLU(inplace=True),
                torch.nn.MaxPool3d(2),
                torch.nn.Conv3d(64, 128, kernel_size=3, padding=1),
                torch.nn.BatchNorm3d(128),
                torch.nn.ReLU(inplace=True),
                torch.nn.MaxPool3d(2),
                torch.nn.Conv3d(128, 256, kernel_size=3, padding=1),
                torch.nn.BatchNorm3d(256),
                torch.nn.ReLU(inplace=True),
                torch.nn.AdaptiveAvgPool3d((1, 1, 1))
            )
            self.classifier = torch.nn.Sequential(
                torch.nn.Dropout(0.5),
                torch.nn.Linear(256, 128),
                torch.nn.ReLU(inplace=True),
                torch.nn.Dropout(0.3),
                torch.nn.Linear(128, num_classes)
            )

        def forward(self, x):
            x = self.conv_layers(x)
            x = x.view(x.size(0), -1)
            return self.classifier(x)

    # Initialize model
    model = Simple3DCNN(num_classes=len(checkpoint['class_names']))
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()

    # Load and preprocess CBCT
    nifti = nib.load(cbct_path)
    volume = nifti.get_fdata().astype(np.float32)
    volume = (volume - volume.min()) / (volume.max() - volume.min() + 1e-8)

    # Resize to 64×64×64
    factors = (64/volume.shape[0], 64/volume.shape[1], 64/volume.shape[2])
    volume = zoom(volume, factors, order=1)
    volume = np.expand_dims(volume, axis=(0, 1))
    volume = torch.from_numpy(volume).float()

    # Predict
    with torch.no_grad():
        outputs = model(volume)
        probs = torch.nn.functional.softmax(outputs, dim=1)

    probs = probs.numpy()[0]
    pred_idx = np.argmax(probs)
    predicted_condition = checkpoint['class_names'][pred_idx]
    confidence = probs[pred_idx]

    # Get top 3 predictions
    top3_idx = np.argsort(probs)[-3:][::-1]
    top3 = [(checkpoint['class_names'][i], probs[i]) for i in top3_idx]

    print(f"Predicted: {predicted_condition} (Confidence: {confidence:.2%})")
    print("\nTop 3 possibilities:")
    for cond, conf in top3:
        print(f"  - {cond}: {conf:.2%}")

    return predicted_condition, confidence, top3

# USAGE - Just provide the path to any CBCT file:
result = load_and_predict('/content/drive/MyDrive/MMDental/MMDental/63/63.nii.gz')